In [ ]:
import os, sys, glob
import os.path as op
sys.path.append('.')
from utils import loadFSL, FSLeyesServer, mkdir_no_exist

import module as lmod
await lmod.purge(force=True)
await lmod.load('fsl/6.0.7.18')
loadFSL()

from fsl.wrappers import fast, flirt
from nilearn import plotting
import matplotlib.pyplot as plt

In [ ]:
data_root = op.abspath(op.join("..", "Data1"))
anat_path = op.join(data_root, "T1", "T1w.nii.gz")
print(op.exists(anat_path))

preproc_root = op.join(data_root, "derivatives", "preprocessed_data")
anat_out = op.join(preproc_root, "anat")
mkdir_no_exist(op.join(data_root, "derivatives"))
mkdir_no_exist(preproc_root)
mkdir_no_exist(anat_out)

fig_dir = "figures"
mkdir_no_exist(fig_dir)

mni_brain = op.expandvars("$FSLDIR/data/standard/MNI152_T1_1mm_brain.nii.gz")

In [ ]:
%gui wx

In [ ]:
fsleyesDisplay = FSLeyesServer()
fsleyesDisplay.show()

In [ ]:
## Skull-stripping
betted_brain = op.join(anat_out, "T1w_brain")
os.system('bet {} {} -m -R'.format(anat_path, betted_brain))

In [ ]:
fsleyesDisplay.resetOverlays()
fsleyesDisplay.load(anat_path)
fsleyesDisplay.load(betted_brain + "_mask")

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(10, 6))
plotting.plot_anat(anat_path, title="T1w original", axes=axes[0])
plotting.plot_anat(betted_brain + ".nii.gz", title="T1w after BET", axes=axes[1])
plt.savefig(op.join(fig_dir, "1a_skullstrip.png"), dpi=200, bbox_inches="tight")

In [ ]:
## Segmentation
segmentation_path = op.join(anat_out, "T1w_fast")
fast(imgs=[betted_brain], out=segmentation_path, n_classes=3)

In [ ]:
fsleyesDisplay.resetOverlays()
fsleyesDisplay.load(betted_brain)
fsleyesDisplay.load(segmentation_path + "_pve_0")
fsleyesDisplay.load(segmentation_path + "_pve_1")
fsleyesDisplay.load(segmentation_path + "_pve_2")
fsleyesDisplay.displayCtx.getOpts(fsleyesDisplay.overlayList[1]).cmap = 'Red'
fsleyesDisplay.displayCtx.getOpts(fsleyesDisplay.overlayList[2]).cmap = 'Green'
fsleyesDisplay.displayCtx.getOpts(fsleyesDisplay.overlayList[3]).cmap = 'Blue'

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(10, 9))
names = ["CSF (pve_0)", "Grey matter (pve_1)", "White matter (pve_2)"]
for i, name in enumerate(names):
    plotting.plot_stat_map(f"{segmentation_path}_pve_{i}.nii.gz",
                           bg_img=betted_brain + ".nii.gz",
                           title=name, cmap="hot", threshold=0.5,
                           colorbar=False, cut_coords=(-30, -20, 10),
                           axes=axes[i])
plt.savefig(op.join(fig_dir, "1b_segmentation.png"), dpi=200, bbox_inches="tight")

In [ ]:
## Registration (linear)
t1_mni = op.join(anat_out, "T1w_brain_mni")
mat = op.join(anat_out, "T1w2mni.mat")
flirt(betted_brain, mni_brain, out=t1_mni, omat=mat)

In [ ]:
fsleyesDisplay.resetOverlays()
fsleyesDisplay.load(mni_brain)
fsleyesDisplay.load(t1_mni)

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(10, 6))
d = plotting.plot_anat(betted_brain + ".nii.gz", title="Before: T1w + MNI contours", axes=axes[0])
d.add_edges(mni_brain)
d = plotting.plot_anat(t1_mni + ".nii.gz", title="After FLIRT: T1w in MNI space + MNI contours", axes=axes[1])
d.add_edges(mni_brain)
plt.savefig(op.join(fig_dir, "1c_registration.png"), dpi=200, bbox_inches="tight")

In [ ]:
## Registration (non linear)
fsl_anat_out = op.join(anat_out, "fsl_anat")
os.system('fsl_anat -i {} -o {} --clobber --nosubcortseg'.format(anat_path, fsl_anat_out))

In [ ]:
mni_head_2mm = op.expandvars("$FSLDIR/data/standard/MNI152_T1_2mm")

fsleyesDisplay.resetOverlays()
fsleyesDisplay.load(mni_head_2mm)
fsleyesDisplay.load(op.join(anat_dir, "T1_to_MNI_lin"))
fsleyesDisplay.load(op.join(anat_dir, "T1_to_MNI_nonlin"))

In [ ]:
mni_brain_2mm = op.expandvars("$FSLDIR/data/standard/MNI152_T1_2mm_brain.nii.gz")
cuts = (-30, -20, 10)

fig, axes = plt.subplots(3, 1, figsize=(10, 9))
d = plotting.plot_anat(anat_path, title="Before registration + MNI contours",
                       cut_coords=cuts, axes=axes[0])
d.add_edges(mni_brain_2mm)
d = plotting.plot_anat(op.join(anat_dir, "T1_to_MNI_lin.nii.gz"), title="Linear (FLIRT) + MNI contours",
                       cut_coords=cuts, axes=axes[1])
d.add_edges(mni_brain_2mm)
d = plotting.plot_anat(op.join(anat_dir, "T1_to_MNI_nonlin.nii.gz"), title="Non-linear (FNIRT) + MNI contours",
                       cut_coords=cuts, axes=axes[2])
d.add_edges(mni_brain_2mm)
plt.savefig(op.join(fig_dir, "1c_registration.png"), dpi=200, bbox_inches="tight")